# Problem 2: dissipation and dispersion

We study the periodic initial-value problem

$$
\begin{equation}
u_t + a u_x = \alpha u_{xx} - \beta u_{xxx},
\qquad -\pi < x < \pi,
\qquad u(x,0)=u_0(x),
\end{equation}
$$

with periodic boundary conditions. This notebook is a notebook version of `scripts/problem2_dissipation_dispersion.py`. It explains the Fourier-mode behavior, compares exact Fourier evolution with a Crank-Nicolson finite-difference discretization, and reproduces the spatial and temporal convergence experiments.

The two parameters play different mathematical roles:

- $\alpha>0$ creates dissipation, meaning Fourier amplitudes decay in time.
- $\beta\ne 0$ creates dispersion, meaning different wave numbers propagate with different speeds.


## Fourier analysis of the continuous problem

For a periodic solution, write

$$
\begin{equation}
u(x,t)=\sum_{k\in\mathbb{Z}} \widehat{u}_k(t)e^{ikx}.
\end{equation}
$$

Then

$$
\frac{\partial}{\partial x}e^{ikx}=ik e^{ikx},
\qquad
\frac{\partial^2}{\partial x^2}e^{ikx}=-k^2 e^{ikx},
\qquad
\frac{\partial^3}{\partial x^3}e^{ikx}=-ik^3 e^{ikx}.
$$

Substitution into the PDE gives the scalar ODE

$$
\begin{equation}
\frac{d\widehat{u}_k}{dt}
=\left(-iak-\alpha k^2+i\beta k^3\right)\widehat{u}_k
=:L_k\widehat{u}_k.
\end{equation}
$$

Therefore

$$
\begin{equation}
\widehat{u}_k(t)=\widehat{u}_k(0)\exp\left[\left(-iak-\alpha k^2+i\beta k^3\right)t\right].
\end{equation}
$$

Equivalently,

$$
\begin{equation}
u(x,t)=\sum_{k\in\mathbb{Z}}\widehat{u}_k(0)
\exp(-\alpha k^2t)
\exp\left(i\left[kx-(ak-\beta k^3)t\right]\right).
\end{equation}
$$

The real exponential $\exp(-\alpha k^2t)$ controls amplitude. When $\alpha>0$, high-frequency modes decay fastest because the decay rate is proportional to $k^2$. This is the smoothing effect of diffusion.

The phase is $kx-\omega(k)t$, where

$$
\begin{equation}
\omega(k)=ak-\beta k^3.
\end{equation}
$$

The phase speed and group speed are

$$
\begin{equation}
c_p(k)=\frac{\omega(k)}{k}=a-\beta k^2,
\qquad
c_g(k)=\omega'(k)=a-3\beta k^2.
\end{equation}
$$

When $\beta\ne 0$, these speeds depend on $k$, so a wave packet spreads because its Fourier components travel at different speeds.


## Finite differences and Crank-Nicolson in Fourier space

Use a uniform periodic grid

$$
\begin{equation}
x_j=-\pi+jh,
\qquad h=\frac{2\pi}{N},
\qquad j=0,1,\ldots,N-1.
\end{equation}
$$

The second-order centered finite-difference symbols for the Fourier mode $e^{ikx_j}$ are

$$
\begin{equation}
\lambda_1(k)=\frac{i\sin(kh)}{h},
\qquad
\lambda_2(k)=\frac{2(\cos(kh)-1)}{h^2},
\qquad
\lambda_3(k)=\frac{i(\sin(2kh)-2\sin(kh))}{h^3}.
\end{equation}
$$

They satisfy

$$
\lambda_1(k)=ik+O(h^2),
\qquad
\lambda_2(k)=-k^2+O(h^2),
\qquad
\lambda_3(k)=-ik^3+O(h^2).
$$

Thus the finite-difference spatial operator has Fourier symbol

$$
\begin{equation}
L_k^{h}= -a\lambda_1(k)+\alpha\lambda_2(k)-\beta\lambda_3(k),
\end{equation}
$$

which is a second-order approximation of $L_k=-iak-\alpha k^2+i\beta k^3$.

Crank-Nicolson applied to $dU_k/dt=L_k^hU_k$ gives

$$
\begin{equation}
\frac{U_k^{n+1}-U_k^n}{\Delta t}
=\frac{1}{2}L_k^h\left(U_k^{n+1}+U_k^n\right),
\end{equation}
$$

so one time step multiplies the mode by

$$
\begin{equation}
G_k=\frac{1+\frac{\Delta t}{2}L_k^h}{1-\frac{\Delta t}{2}L_k^h}.
\end{equation}
$$

The script uses this diagonal Fourier representation of the periodic finite-difference matrices. That is why the implementation is compact: each Fourier coefficient evolves independently.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 4.8),
    "axes.grid": True,
    "lines.linewidth": 2,
})


def find_project_root(start=None):
    """Find the repository root whether the notebook is launched from root or notebooks/."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts").is_dir() and (candidate / "figures").is_dir():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
FIGURES_DIR = PROJECT_ROOT / "figures" / "problem2"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def relative_to_root(path):
    return Path(path).resolve().relative_to(PROJECT_ROOT)


## Exact Fourier evolution on the grid

The function below takes the discrete Fourier transform of the initial data, evolves each mode by the exact continuous multiplier $\exp(L_k t)$, and transforms back. For smooth periodic data on this grid, this is the reference solution used throughout the script.


In [ ]:
def exact_solution_fft(x, t, u0, a, alpha, beta):
    """Exact Fourier evolution for periodic data sampled on a uniform grid."""
    N = len(x)
    h = 2.0 * np.pi / N
    U0 = np.fft.fft(u0.astype(complex))
    kk = np.fft.fftfreq(N, d=h / (2.0 * np.pi))

    Lk = -1j * a * kk - alpha * kk**2 + 1j * beta * kk**3
    return np.fft.ifft(U0 * np.exp(Lk * t)).real


## Crank-Nicolson finite-difference solver

The original script advances by repeatedly multiplying by the Crank-Nicolson factor $G_k$. Because the multiplier is constant in time for this linear constant-coefficient problem,

$$
\begin{equation}
U_k^{n}=G_k^nU_k^0.
\end{equation}
$$

The notebook uses this algebraically equivalent power form, which preserves the same numerical scheme while keeping the convergence cells responsive.


In [ ]:
def fd_cn_periodic(x, T, u0, a, alpha, beta, dt):
    """
    Second-order Crank-Nicolson time integration with second-order centered
    finite-difference spatial operators on a periodic grid.
    """
    N = len(x)
    h = 2.0 * np.pi / N
    kk = np.fft.fftfreq(N, d=h / (2.0 * np.pi))

    lam1 = 1j * np.sin(kk * h) / h
    lam2 = 2.0 * (np.cos(kk * h) - 1.0) / h**2
    lam3 = 1j * (np.sin(2.0 * kk * h) - 2.0 * np.sin(kk * h)) / h**3

    Lk = -a * lam1 + alpha * lam2 - beta * lam3
    cn_mult = (1.0 + 0.5 * dt * Lk) / (1.0 - 0.5 * dt * Lk)

    Nsteps = int(round(T / dt))
    if not np.isclose(Nsteps * dt, T):
        raise ValueError("T must be an integer multiple of dt for this experiment.")

    U = np.fft.fft(u0.astype(complex))
    U *= cn_mult**Nsteps
    return np.fft.ifft(U).real


## Illustration: advection, dissipation, and dispersion

We use the smooth periodic initial condition

$$
\begin{equation}
u_0(x)=\sin x+0.6\cos(2x)+0.3\sin(3x).
\end{equation}
$$

The three comparison cases are:

- pure advection: $\alpha=0$, $\beta=0$;
- dissipation: $\alpha=0.15$, $\beta=0$;
- dispersion: $\alpha=0$, $\beta=0.05$.

In the dissipative case, the oscillations are damped, especially the higher-frequency components. In the dispersive case, the shape changes because the three Fourier modes travel with different phase speeds.


In [ ]:
N = 512
x = np.linspace(-np.pi, np.pi, N, endpoint=False)
a_val = 1.0
T_ill = 2.0
dt = 1e-3


def u0_smooth(x):
    return np.sin(x) + 0.6 * np.cos(2.0 * x) + 0.3 * np.sin(3.0 * x)


u0 = u0_smooth(x)

cases = [
    (0.0, 0.0, r"Pure advection: $\alpha=0,\ \beta=0$"),
    (0.15, 0.0, r"Dissipation: $\alpha=0.15,\ \beta=0$"),
    (0.0, 0.05, r"Dispersion: $\alpha=0,\ \beta=0.05$"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (alpha, beta, title) in zip(axes, cases):
    u_ex = exact_solution_fft(x, T_ill, u0, a_val, alpha, beta)
    u_fd = fd_cn_periodic(x, T_ill, u0, a_val, alpha, beta, dt)

    ax.plot(x, u0, "g--", lw=1.5, label=r"Initial $u_0$")
    ax.plot(x, u_ex, "b-", lw=2, label=fr"Exact Fourier, $t={T_ill}$")
    ax.plot(x, u_fd, "r:", lw=2, label="FD Crank-Nicolson")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(r"$x$")
    ax.legend(fontsize=8)

fig.suptitle(r"$u_t+u_x=\alpha u_{xx}-\beta u_{xxx}$, $a=1$", fontsize=13)
fig.tight_layout()
dissipation_dispersion_path = FIGURES_DIR / "dissipation_dispersion.png"
fig.savefig(dissipation_dispersion_path, dpi=120)
print(f"Saved: {relative_to_root(dissipation_dispersion_path)}")
plt.show()


## Spatial convergence experiment

To isolate the spatial error, the script uses a very small time step and varies $N$. The measured error is

$$
\begin{equation}
\|u_h(T)-u(T)\|_\infty
=\max_j |u_h(x_j,T)-u(x_j,T)|.
\end{equation}
$$

Since the spatial finite differences are second order, the expected spatial scaling is

$$
\begin{equation}
\|u_h(T)-u(T)\|_\infty \approx C h^2,
\qquad h=\frac{2\pi}{N}.
\end{equation}
$$


In [ ]:
alpha_c, beta_c = 0.1, 0.05
T_c = 0.5
dt_ref = 1e-6

print("Spatial convergence test (fixed dt=1e-6, vary N):")
Ns_sp = [16, 32, 64, 128, 256]
errs_sp = []

for Ni in Ns_sp:
    xi = np.linspace(-np.pi, np.pi, Ni, endpoint=False)
    u0i = u0_smooth(xi)
    u_ex = exact_solution_fft(xi, T_c, u0i, a_val, alpha_c, beta_c)
    u_fd = fd_cn_periodic(xi, T_c, u0i, a_val, alpha_c, beta_c, dt_ref)
    err = np.max(np.abs(u_fd - u_ex))
    errs_sp.append(err)
    print(f"  N={Ni:4d}  h={2*np.pi/Ni:.4f}  err={err:.3e}")

errs_sp = np.array(errs_sp)
rates_sp = np.log2(errs_sp[:-1] / errs_sp[1:])
print("  Convergence rates:", np.round(rates_sp, 3))


## Temporal convergence experiment with finite differences in space

Next, the script fixes a fine grid and varies $\Delta t$. Crank-Nicolson is a second-order time integrator, so before spatial error dominates we expect

$$
\begin{equation}
\|u_{\Delta t}(T)-u(T)\|_\infty \approx C(\Delta t)^2.
\end{equation}
$$

Because the comparison is still made against the exact continuous Fourier solution, the finite-difference spatial error remains present. On a sufficiently fine grid this is small, but it can still create an error floor for the smallest time steps.


In [ ]:
N_t = 512
x_t = np.linspace(-np.pi, np.pi, N_t, endpoint=False)
u0_t = u0_smooth(x_t)
u_ex_t = exact_solution_fft(x_t, T_c, u0_t, a_val, alpha_c, beta_c)

print("Temporal convergence test (fixed N=512, vary dt):")
dts = [0.1, 0.05, 0.025, 0.0125]
errs_dt = []

for dti in dts:
    u_fd = fd_cn_periodic(x_t, T_c, u0_t, a_val, alpha_c, beta_c, dti)
    err = np.max(np.abs(u_fd - u_ex_t))
    errs_dt.append(err)
    print(f"  dt={dti:.4f}  err={err:.3e}")

errs_dt = np.array(errs_dt)
rates_dt = np.log2(errs_dt[:-1] / errs_dt[1:])
print("  Convergence rates:", np.round(rates_dt, 3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

hs = np.array([2.0 * np.pi / Ni for Ni in Ns_sp])
axes[0].loglog(hs, errs_sp, "bo-", label="FD max error")
axes[0].loglog(hs, errs_sp[2] * (hs / hs[2])**2, "k--", label=r"$O(h^2)$ reference")
axes[0].set_xlabel(r"$h=2\pi/N$")
axes[0].set_ylabel("Max error")
axes[0].set_title("Spatial convergence")
axes[0].legend()

axes[1].loglog(dts, errs_dt, "ro-", label="FD max error")
axes[1].loglog(dts, errs_dt[1] * (np.array(dts) / dts[1])**2, "k--", label=r"$O(\Delta t^2)$ reference")
axes[1].set_xlabel(r"$\Delta t$")
axes[1].set_ylabel("Max error")
axes[1].set_title("Temporal convergence")
axes[1].legend()

fig.suptitle("Convergence of CN-FD for the dissipation-dispersion PDE", fontsize=12)
fig.tight_layout()
convergence_fd_path = FIGURES_DIR / "convergence_fd.png"
fig.savefig(convergence_fd_path, dpi=120)
print(f"Saved: {relative_to_root(convergence_fd_path)}")
plt.show()


## Temporal convergence with an exact spectral spatial operator

The final script experiment removes the finite-difference spatial error by using the exact Fourier symbol $L_k$ in the Crank-Nicolson update. Then the only systematic discretization error is the time-stepping error.

For the spectral-in-space Crank-Nicolson method,

$$
\begin{equation}
G_k^{\mathrm{spec}}
=\frac{1+\frac{\Delta t}{2}L_k}{1-\frac{\Delta t}{2}L_k},
\qquad
L_k=-iak-\alpha k^2+i\beta k^3.
\end{equation}
$$

This gives a cleaner second-order temporal convergence plot.


In [ ]:
def spectral_cn(x, T, u0, a, alpha, beta, dt):
    """Crank-Nicolson in time with the exact spectral spatial operator."""
    N = len(x)
    h = 2.0 * np.pi / N
    kk = np.fft.fftfreq(N, d=h / (2.0 * np.pi))

    Lk = -1j * a * kk - alpha * kk**2 + 1j * beta * kk**3
    cn_mult = (1.0 + 0.5 * dt * Lk) / (1.0 - 0.5 * dt * Lk)

    Nsteps = int(round(T / dt))
    if not np.isclose(Nsteps * dt, T):
        raise ValueError("T must be an integer multiple of dt for this experiment.")

    U = np.fft.fft(u0.astype(complex))
    U *= cn_mult**Nsteps
    return np.fft.ifft(U).real


N_t2 = 256
x_t2 = np.linspace(-np.pi, np.pi, N_t2, endpoint=False)
u0_t2 = u0_smooth(x_t2)
u_ex_t2 = exact_solution_fft(x_t2, T_c, u0_t2, a_val, alpha_c, beta_c)

print("Temporal convergence (spectral CN, N=256):")
dts2 = [0.1, 0.05, 0.025, 0.0125, 0.00625]
errs_dt2 = []

for dti in dts2:
    u_fd = spectral_cn(x_t2, T_c, u0_t2, a_val, alpha_c, beta_c, dti)
    err = np.max(np.abs(u_fd - u_ex_t2))
    errs_dt2.append(err)
    print(f"  dt={dti:.5f}  err={err:.3e}")

errs_dt2 = np.array(errs_dt2)
rates_dt2 = np.log2(errs_dt2[:-1] / errs_dt2[1:])
print("  Convergence rates:", np.round(rates_dt2, 3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

hs2 = np.array([2.0 * np.pi / Ni for Ni in Ns_sp])
axes[0].loglog(hs2, errs_sp, "bo-", label="FD max error")
axes[0].loglog(hs2, errs_sp[2] * (hs2 / hs2[2])**2, "k--", label=r"$O(h^2)$ reference")
axes[0].set_xlabel(r"$h=2\pi/N$")
axes[0].set_ylabel("Max error")
axes[0].set_title("Spatial convergence, centered FD")
axes[0].legend()

axes[1].loglog(dts2, errs_dt2, "rs-", label="Spectral-CN max error")
axes[1].loglog(dts2, errs_dt2[1] * (np.array(dts2) / dts2[1])**2, "k--", label=r"$O(\Delta t^2)$ reference")
axes[1].set_xlabel(r"$\Delta t$")
axes[1].set_ylabel("Max error")
axes[1].set_title("Temporal convergence, spectral CN")
axes[1].legend()

fig.suptitle("Second-order convergence in space and time", fontsize=12)
fig.tight_layout()
convergence_spectral_path = FIGURES_DIR / "convergence_spectral.png"
fig.savefig(convergence_spectral_path, dpi=120)
print(f"Saved: {relative_to_root(convergence_spectral_path)}")
plt.show()
